### Ensemble Retriever & Hybrid Search (EnsembleRetriever)

Hybrid Search combines sparse keyword-based search (BM25Retriever) with dense vector-based search (Chroma) to deliver superior retrieval accuracy.

EnsembleRetriever uses the Reciprocal Rank Fusion (RRF) algorithm to rerank and merge search results across multiple retrievers.

### Why Hybrid Search Superiority

* BM25 Sparse Search: Excels at exact keyword matches, technical part numbers, acronyms, and proper nouns.
* Dense Vector Search: Excels at semantic meaning, paraphrased concepts, and broad query similarity.
* Ensemble Weights: Configurable weighting parameters (e.g. weights=[0.5, 0.5]) control the balance between sparse and dense search scores.

In [1]:
from dotenv import load_dotenv, find_dotenv
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

load_dotenv(find_dotenv())

docs = [
    Document(page_content="Model GTX-9000 is an advanced industrial water filtration pump.", metadata={"type": "spec"}),
    Document(page_content="Clean water systems utilize mechanical filters to remove particulate matter.", metadata={"type": "general"}),
    Document(page_content="High pressure hydraulic pumps require quarterly maintenance and oil checks.", metadata={"type": "maintenance"})
]

# 1. Initialize Sparse Keyword BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

# 2. Initialize Dense Vector Retriever (Ephemeral in-memory Chroma)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = Chroma.from_documents(docs, embeddings)
chroma_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Combine in EnsembleRetriever with Reciprocal Rank Fusion
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)

# Query containing exact keyword model number
hybrid_results = ensemble_retriever.invoke("GTX-9000 pump maintenance")

print(f"EnsembleRetriever returned {len(hybrid_results)} hybrid search result(s):")
for i, doc in enumerate(hybrid_results, 1):
    print(f"Doc {i}: {doc.page_content}")


/var/folders/0h/sdy0_vy9385bh841gfp_jzdm0000gn/T/ipykernel_48603/3640115269.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EnsembleRetriever returned 2 hybrid search result(s):
Doc 1: Model GTX-9000 is an advanced industrial water filtration pump.
Doc 2: High pressure hydraulic pumps require quarterly maintenance and oil checks.
